# Real SAEs: the feature-feature condition (eq. 6) on and off the training distribution

We test the feature-feature condition on the GPT2-small block-8 SAEs from SAE-lens. That's the main 24,576-latent `gpt2-small-res-jb` SAE, plus the 768–6,144-latent `gpt2-small-res-jb-feature-splitting` SAEs from the same training recipe. All of them were trained on OpenWebText.

**The claim.** The condition must hold on the training distribution. Off-distribution it can fail, and the theory says how: by hierarchies the SAE never had to split. We show:
1. **Pairs:** no violations on OpenWebText (two disjoint samples), but a systematic drift towards violation as the data moves off-distribution. Robust strict hierarchies appear on a narrow domain (Python code).
2. **Beyond pairs:** on code, *sets* of features violate the condition jointly even though every pair within them passes. These are collective soft hierarchies.
3. **Nulls:** for these sparse, nearly orthogonal features, the condition is insensitive to the decoder cosines, and constrains the joint activity instead.

**Coordinates.** $u_d = z_d/\mathrm{mean}(z_d)$, which is the paper's $\bar z/|\min \bar z|$ plus one, so it is 0 exactly when the feature is off. Eq. (6) projected onto a set S of other features is $1-\cos(\hat w_d,\hat w_S)\in\mathrm{conv}\{u_S^{[i]}: z_d^{[i]}=0\}$. We summarise it by the **hull gauge** $g(d,S)$, the smallest total weight on the tokens where d is off needed to reach the target point, with the remaining weight on tokens where every feature in S is off too. The condition holds iff $g\le 1$, and $g=\infty$ means a strict hierarchy. For a single feature $d'$, $g = (1-\cos\theta_{dd'})/\max_{A_d}u_{d'}$. See `sae_conditions.py`.

**Inputs** (see README): `extract_sae_activations.py`, `dataset_shift.py`, `joint_violations.py`, `run_subset_sweep.py`.

In [ ]:
import json, os
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sae_conditions import FeatureData

CACHE = os.path.expanduser('~/sae_identifiability_cache')
SAES = ['fs_768', 'fs_1536', 'fs_3072', 'fs_6144', 'main']            # ordered by width
WIDTH = {'fs_768': 768, 'fs_1536': 1536, 'fs_3072': 3072, 'fs_6144': 6144, 'main': 24576}
WIDTH_COLORS = ['#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#0d366b']  # sequential: light = narrow
CAT = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']                     # categorical, fixed order
LABEL = {'openwebtext': 'OWT', 'openwebtext_b': 'OWT (2nd sample)', 'c4': 'C4', 'wiki': 'Wikipedia',
         'pile': 'Pile', 'math': 'Math', 'wiki_de': 'German wiki', 'wiki_fr': 'French wiki',
         'code': 'Code', 'code_python': 'Python code'}

def clean(ax):
    ax.spines[['top', 'right']].set_visible(False)

shift = {s: json.load(open(f'{CACHE}/dataset_shift_{s}.json')) for s in SAES if os.path.exists(f'{CACHE}/dataset_shift_{s}.json')}

## 1. Pairwise condition across datasets and SAE widths

Each row is one (SAE, dataset). The columns are:
* **FVU**: the SAE's fraction of variance unexplained, a proxy for how far off-distribution the data is.
* **F**: the number of features active on more than 0.5% of tokens.
* **violations**: ordered pairs violating eq. 6, broken down by type.
* **replicate**: how many of those violations hold in both halves of the tokens.
* **g > 0.1**: the number of pairs within 10× of violating.

In [ ]:
print(f"{'SAE':>6s} {'dataset':>16s} {'FVU':>6s} {'F':>6s} {'viol':>5s} {'strict':>6s} {'never-off':>9s} {'replic.':>7s} {'g>0.1':>7s} {'max g':>6s}")
for s in SAES:
    for ds, r in shift.get(s, {}).items():
        m = r['summary']
        print(f"{WIDTH[s]:6d} {LABEL.get(ds, ds):>16s} {m['fvu']:6.3f} {m['n_features']:6d} {m['n_violations']:5d} "
              f"{m['n_strict_hierarchy']:6d} {m['n_never_coinactive']:9d} {m['n_replicating']:7d} "
              f"{m['n_pairs_gauge_above_0p1']:7d} {m['gauge_max_finite']:6.3f}")

In [ ]:
# Drift towards violation vs. distance from the training distribution
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for s, col in zip(SAES, WIDTH_COLORS):
    rs = shift.get(s, {})
    if not rs: continue
    names = sorted(rs, key=lambda d: rs[d]['summary']['fvu'])
    fvu = [rs[d]['summary']['fvu'] for d in names]
    for ax, key in zip(axs, ['gauge_max_finite', 'n_pairs_gauge_above_0p1']):
        y = [rs[d]['summary'][key] for d in names]
        ax.plot(fvu, np.maximum(y, 0.5) if key.startswith('n_') else y, '-o', color=col, lw=2, ms=6,
                markeredgecolor='white', markeredgewidth=1.5, label=f'{WIDTH[s]} latents')
    if s == 'main':
        for d, x, y in zip(names, fvu, [rs[d]['summary']['gauge_max_finite'] for d in names]):
            axs[0].annotate(LABEL.get(d, d), (x, y), xytext=(4, 4), textcoords='offset points', fontsize=8, color='#555')
axs[0].axhline(1, color='k', ls=':', lw=1); axs[0].text(axs[0].get_xlim()[0], 1.05, 'violation', fontsize=9)
axs[0].set(xlabel='SAE FVU on dataset (distance from training distribution)', ylabel='largest finite pairwise gauge', yscale='log')
axs[1].set(xlabel='SAE FVU on dataset', ylabel='pairs with gauge > 0.1 (0 plotted at 0.5)', yscale='log')
axs[1].legend(frameon=False, fontsize=9)
for ax in axs: clean(ax)
plt.tight_layout()

### Strict hierarchies on a narrow domain

On Python code, the main SAE has features that fire only when a broader "inside indented code" feature is also firing. Those broader features are active on about 30% of Python tokens and almost never on OWT. Every one of these violations replicates in both halves of the tokens.

In [ ]:
viol = shift['main']['code_python']['violations']
for v in viol:
    print(f"parent {v['d']:6d} (freq {v['freq_d']:.3f}, OWT {v['freq_d_owt']:.4f})  child {v['d_prime']:6d} "
          f"(n_on {v['n_on_d_prime']}, OWT {v['freq_d_prime_owt']:.4f})  cos {v['cos']:+.3f}  replicates {v['replicates']}")

# Fig 1H-style picture of the largest one: child activity against parent activity, paper's coordinates
v = max(viol, key=lambda v: v['n_on_d_prime'])
Z = sp.load_npz(f'{CACHE}/code_python/feature_acts.npz').tocsc()
W = np.load(f'{CACHE}/openwebtext/sae_params.npz')['W_dec']
parent, child = [Z[:, f].toarray().ravel() for f in (v['d'], v['d_prime'])]
norm = lambda z: (z - z.mean()) / z.mean()          # = z_bar / |min z_bar| when min z = 0
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(norm(parent), norm(child), s=6, color=CAT[0], alpha=0.4, label='tokens')
ax.scatter([-1], [-v['cos']], s=70, color=CAT[1], edgecolor='white', linewidth=2, zorder=3,
           label=r'required: $-\cos\theta$ on the parent-off line')
ax.set(xlabel=f"parent feature {v['d']}", ylabel=f"child feature {v['d_prime']}",
       title='strict hierarchy on Python code: child never fires while parent is off')
ax.legend(frameon=False, fontsize=8, loc='upper left'); clean(ax)
plt.tight_layout()

## 2. Beyond pairs: collective hierarchies

For the 25 features d whose pairwise constraints are closest to binding, we test eq. 6 jointly against the k features that individually come closest to violating. On code, sets of 20–30 features violate the condition together, even though every pair within the set passes easily. Each member fires mostly while d is on, rarely and weakly while d is off, and not together, so no weighting of the d-off tokens can reach the target.

In [ ]:
joint = json.load(open(f'{CACHE}/joint_violations_main.json'))
fig, ax = plt.subplots(figsize=(6, 4))
for ds, col in zip([d for d in ['openwebtext', 'pile', 'code', 'code_python'] if d in joint], CAT):
    ks = sorted(joint[ds]['gauges'], key=int)
    g = np.array([joint[ds]['gauges'][k] for k in ks])
    g_plot = np.where(np.isinf(g), 5.0, g)            # infinite gauges drawn at the top edge
    x = [int(k) for k in ks]
    ax.plot(x, np.median(g_plot, 1), '-', color=col, lw=2, label=f'{LABEL[ds]} (median)')
    ax.plot(x, np.max(g_plot, 1), '--', color=col, lw=1.5, label=f'{LABEL[ds]} (max)')
ax.axhline(1, color='k', ls=':', lw=1); ax.text(1, 1.1, 'violation', fontsize=9)
ax.set(xscale='log', yscale='log', ylim=(0.01, 6), xlabel='subset size k (worst pairs)', ylabel='hull gauge g(d, S)',
       title='joint condition, 25 most constrained features')
ax.legend(frameon=False, fontsize=8, ncol=2); clean(ax)
plt.tight_layout()
for ds in joint:
    for v in joint[ds]['violations']:
        print(f"{LABEL.get(ds, ds)}: feature {v['d']} violates at k={v['k']} (gauge {v['gauge']:.2f}); minimal set of "
              f"{len(v['minimal_S'])} features, worst pair in it {v['max_pair_gauge_in_S']:.3f}; gauges on the two halves "
              f"{[None if h is None else round(h, 2) for h in v['half_gauges']]}")

In [ ]:
# Anatomy of one collective hierarchy: how each member of the minimal set behaves relative to d
ds = 'code' if joint.get('code', {}).get('violations') else next(d for d in joint if joint[d]['violations'])
v = max(joint[ds]['violations'], key=lambda v: v['minimal_gauge'])
Z = sp.load_npz(f'{CACHE}/{ds}/feature_acts.npz').tocsc()
on_d = Z[:, v['d']].toarray().ravel() > 0
p_on, p_off = [], []
for f in v['minimal_S']:
    on = Z[:, f].toarray().ravel() > 0
    p_on.append(on[on_d].mean()); p_off.append(on[~on_d].mean())
order = np.argsort(p_on)[::-1]
fig, ax = plt.subplots(figsize=(8, 3.5))
x = np.arange(len(order))
ax.bar(x - 0.2, np.array(p_on)[order], 0.4, color=CAT[0], label=f"P(member on | feature {v['d']} on)")
ax.bar(x + 0.2, np.array(p_off)[order], 0.4, color=CAT[1], label=f"P(member on | feature {v['d']} off)")
ax.set_xticks(x, [v['minimal_S'][i] for i in order], rotation=90, fontsize=7)
ax.set(ylabel='probability', xlabel='members of the minimal violating set',
       title=f"{LABEL.get(ds, ds)}: collective hierarchy under feature {v['d']} (gauge {v['minimal_gauge']:.2f})")
ax.legend(frameon=False, fontsize=8); clean(ax)
plt.tight_layout()

## 3. What the condition is sensitive to: two nulls

`run_subset_sweep.py` takes 150 random target features and, for several ways of choosing S, compares the real gauge with two nulls:
* **cos null:** d's decoder cosines are replaced by those of a random other feature, keeping the activity.
* **perm null:** each feature's activity is shuffled independently across tokens, keeping the marginals and the decoder but destroying co-activation.

The real gauge matches the cos null almost exactly: for sparse, nearly orthogonal features the target $1-\cos \approx 1$ whatever the cosines are, so the condition is really a constraint on the joint activity. Against the perm null, the real gauge is 20–40% lower for structured subsets, so the SAE's co-activation structure sits more comfortably inside the condition than independent activity would.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, null, title in zip(axs, ['cos', 'perm'], ['shuffled-cosine null', 'activity-permutation null']):
    path = f'{CACHE}/openwebtext/subset_sweep_{null}.npz'
    if not os.path.exists(path): continue
    sw = np.load(path)
    res, ks = sw['results'], sw['ks']
    for si, (s, col) in enumerate(zip(sw['strategies'], CAT)):
        ax.plot(ks, np.median(res[:, si, :, 0] / res[:, si, :, 1], axis=0), '-o', color=col, lw=2, ms=5,
                markeredgecolor='white', markeredgewidth=1.5, label=s)
    ax.axhline(1, color='k', ls=':', lw=1)
    ax.set(xscale='log', xlabel='subset size k', title=title); clean(ax)
axs[0].set_ylabel('median  g_real / g_null'); axs[0].legend(frameon=False, fontsize=9)
plt.tight_layout()

## 4. Interpreting the violations

**Python code.** GPT-2 tokenises each indentation space as its own `' '` token, so 31% of Python tokens are indentation. The figure shows, for the features in the Python violations, the probability of firing on the n-th space of a run. The parents ("indentation space") fire on essentially every space from the 3rd. The child 16719 ("deep indentation") never fires on the 1st space and rises to 85% by the 8th. It is a strict subset of the parents. The members of the collective hierarchies tile positions within the run.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2')
strs = [tok.decode([t]) for t in range(tok.vocab_size)]
is_space = np.array([s.strip() == '' and '\n' not in s for s in strs])

def position_in_run(tokens, ctx=127):
    # 1 for the first whitespace token after a non-whitespace token, 2 for the next, ...; 0 otherwise
    pos = np.zeros(len(tokens), int)
    for s in range(len(tokens) // ctx):
        p = 0
        for i in range(s * ctx, (s + 1) * ctx):
            p = p + 1 if is_space[tokens[i]] else 0
            pos[i] = p
    return pos

tokens = np.load(f'{CACHE}/code_python/tokens.npy')
pos = position_in_run(tokens)
Z = sp.load_npz(f'{CACHE}/code_python/feature_acts.npz').tocsc()
groups = {'parents ("indentation space")': [9858, 9303, 22776],
          'child ("deep indentation")': [16719],
          'members of a collective hierarchy': [10155, 1590, 17178, 1278]}
P = np.arange(1, 13)
fig, ax = plt.subplots(figsize=(7, 4.5))
for (name, feats), col in zip(groups.items(), CAT):
    for j, f in enumerate(feats):
        on = Z[:, f].toarray().ravel() > 0
        ax.plot(P, [on[pos == p].mean() for p in P], '-o', color=col, lw=2, ms=5, alpha=0.9,
                label=name if j == 0 else None)
ax.set(xlabel='position within indentation run (n-th space)', ylabel='P(feature fires)', title='Python code, main SAE')
ax.legend(frameon=False, fontsize=9); clean(ax)
plt.tight_layout()

**German and French.** Features that on OWT respond to foreign words inside English text fire on 75–95% of German or French tokens, and contain generic grammar features ("that/which/who", "as/like", sentence-final newlines, commas) that fire on the German or French counterparts.

In [ ]:
for sae, name in [('main', ''), ('fs_768', 'fs_768'), ('fs_3072', 'fs_3072')]:
    for ds in ['wiki_de', 'wiki_fr']:
        if sae not in shift or ds not in shift[sae]: continue
        toks = np.load(f'{CACHE}/{ds}/tokens.npy'); Z = sp.load_npz(f'{CACHE}/{ds}/{name}/feature_acts.npz').tocsc()
        for v in shift[sae][ds]['violations']:
            z = Z[:, v['d_prime']].toarray().ravel()
            ctxs = [tok.decode(toks[max(i - 4, 0):i + 1]).replace('\n', ' ') for i in np.argsort(-z)[:4]]
            print(f"{WIDTH[sae]:6d} {LABEL[ds]:12s} parent {v['d']} (on {v['freq_d']:.0%} here, {v['freq_d_owt']:.2%} of OWT) "
                  f"contains child {v['d_prime']}: {ctxs}")

## 5. What the violating features do on OpenWebText

For every pairwise violation we look up the same two features on both OWT samples (130k tokens). The hierarchies are not there: the child fires almost always with the parent off, and often has a different role altogether (an "island" feature becomes "deep indentation" on Python code; an English "that/which/who" feature fires on German relative pronouns). The new data creates the hierarchy by making one feature ubiquitous and giving others new roles inside it.

In [ ]:
import collections
owt_tokens = np.concatenate([np.load(f'{CACHE}/{d}/tokens.npy') for d in ['openwebtext', 'openwebtext_b']])
for sae, sub in [('main', ''), ('fs_768', 'fs_768'), ('fs_3072', 'fs_3072')]:
    Z_owt = sp.vstack([sp.load_npz(f'{CACHE}/{d}/{sub}/feature_acts.npz') for d in ['openwebtext', 'openwebtext_b']]).tocsc()
    seen = set()
    for ds, r in shift.get(sae, {}).items():
        for v in r['violations']:
            if (v['d'], v['d_prime']) in seen: continue
            seen.add((v['d'], v['d_prime']))
            parent, child = [Z_owt[:, f].toarray().ravel() > 0 for f in (v['d'], v['d_prime'])]
            common = lambda on: [t for t, _ in collections.Counter(tok.decode([t]) for t in owt_tokens[on]).most_common(4)]
            print(f"{WIDTH[sae]:6d} {LABEL[ds]:12s} parent {v['d']:6d} (OWT: {parent.sum():4d} tokens, e.g. {common(parent)})  "
                  f"child {v['d_prime']:6d} (OWT: {child.sum():4d} tokens, {(child & ~parent).sum()} with parent off, e.g. {common(child)})")

## 6. Feature-residual condition (eq. 8)

For each feature d and each of the top 50 principal directions u of the residuals, `residual_condition.py` computes the smallest λ at which eq. (8) holds, and λ* is the maximum over pairs. The table gives λ*/λ_train, where λ_train is the SAE's training L1 coefficient converted to the paper's convention (App. D.7), so values > 1 violate.

In [ ]:
R = json.load(open(f'{CACHE}/residual_condition.json'))
rsae = ['fs_768', 'fs_1536', 'fs_3072', 'fs_6144', 'main_24576']
dss = [d for d in LABEL if any(f'{s}/{d}' in R for s in rsae)]
print(f"{'dataset':>14s}" + ''.join(f'{s.split("_")[-1]:>8s}' for s in rsae))
for d in dss:
    print(f'{LABEL[d]:>14s}' + ''.join(f"{R[f'{s}/{d}']['lam_star'] / R[f'{s}/{d}']['lam_train']:8.2f}" if f'{s}/{d}' in R else f"{'-':>8s}" for s in rsae))
r = R['fs_768/code']
print('\nCode, 768 latents: residual PC1 sd', round(r['pc_sd'][0], 1), '; its two ends:')
print('  +', r['pc_top_tokens']['0']['positive'][:6]); print('  -', r['pc_top_tokens']['0']['negative'][:6])
print('violating features (all on PC1):', sorted({v['feature'] for v in r['violations']}))